# π₀.₅ (Pi0.5) — Vision-Language-Action Policy on AMD ROCm

**Platform:** AMD ROCm (gfx1100, 48 GB VRAM)  
**Model:** [`lerobot/pi05_base`](https://huggingface.co/lerobot/pi05_base)  
**Framework:** [LeRobot](https://github.com/huggingface/lerobot) by Hugging Face  
**Architecture:** Vision-Language-Action (VLA) — open-world generalization  
**License:** Apache 2.0

π₀.₅ is Physical Intelligence's open-world robot policy, adapted in LeRobot. It co-trains on heterogeneous data (web images, verbal instructions, cross-embodiment robot demos) to generalize across environments, objects, and tasks never seen during training.

---

## Hardware Requirements

| Component | Requirement | This machine |
|---|---|---|
| GPU VRAM | ~24 GB (BF16, `train_expert_only`) | ✅ 48 GB gfx1100 |
| GPU VRAM | ~40 GB (BF16, full finetune) | ✅ fits |
| BF16 support | Required | ✅ RDNA3 native |
| Python | ≥ 3.12 | check below |

## 0. Environment Verification

Verify the ROCm stack and Python version. On AMD hardware, PyTorch exposes HIP through the standard `torch.cuda` API.

In [ ]:
import subprocess, sys, os

print("=" * 55)
print("ROCm & GPU Environment")
print("=" * 55)

result = subprocess.run(
    ["rocm-smi", "--showproductname", "--showmeminfo", "vram"],
    capture_output=True, text=True,
)
print(result.stdout if result.returncode == 0 else "rocm-smi not found — skipping")

import torch
print(f"Python version   : {sys.version.split()[0]}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA (HIP) avail : {torch.cuda.is_available()}")
print(f"ROCm/HIP version : {getattr(torch.version, 'hip', 'N/A')}")
print(f"GPU count        : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    vram_gb = props.total_memory / (1024**3)
    print(f"  GPU {i}: {props.name}  |  VRAM: {vram_gb:.1f} GB")

total_vram = sum(
    torch.cuda.get_device_properties(i).total_memory
    for i in range(torch.cuda.device_count())
) / (1024**3)
print(f"\nTotal VRAM: {total_vram:.1f} GB")

bf16_ok = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
print(f"BF16 supported   : {bf16_ok}")

if total_vram >= 24:
    print("✅ Sufficient VRAM for π₀.₅ (expert-only finetune, ~24 GB)")
else:
    print("⚠️  Less than 24 GB VRAM — use train_expert_only=true and gradient_checkpointing")

import sys
major, minor = sys.version_info[:2]
if major == 3 and minor >= 12:
    print("✅ Python 3.12+ — LeRobot requirement satisfied")
else:
    print(f"⚠️  Python {major}.{minor} — LeRobot requires Python ≥ 3.12")

## 1. ROCm Environment Variables

Set these before importing torch or lerobot. They improve performance and compatibility on RDNA3/CDNA GPUs.

In [ ]:
import os, subprocess

# Better HIP kernel error messages during debugging
os.environ["TORCH_USE_HIP_DSA"] = "1"

# AMD Triton-based Flash Attention (RDNA3 gfx1100 supports this)
os.environ["FLASH_ATTENTION_TRITON_AMD_ENABLE"] = "TRUE"

# Needed for torch.compile on ROCm — avoids hipblaslt fallback issues
os.environ["PYTORCH_TUNABLEOP_ENABLED"] = "1"

# If gfx1100 kernel images are missing for a package, override the GFX version
# os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.0.0"

# Restrict to GPU 0 (ROCm equivalent of CUDA_VISIBLE_DEVICES)
os.environ["ROCR_VISIBLE_DEVICES"] = "0"

# Disable NUMA balancing (recommended on multi-NUMA systems)
subprocess.run(
    ["sudo", "sh", "-c", "echo 0 > /proc/sys/kernel/numa_balancing"],
    capture_output=True,
)

print("ROCm environment variables set.")
print(f"  TORCH_USE_HIP_DSA                 = {os.environ['TORCH_USE_HIP_DSA']}")
print(f"  FLASH_ATTENTION_TRITON_AMD_ENABLE  = {os.environ['FLASH_ATTENTION_TRITON_AMD_ENABLE']}")
print(f"  PYTORCH_TUNABLEOP_ENABLED          = {os.environ['PYTORCH_TUNABLEOP_ENABLED']}")
print(f"  ROCR_VISIBLE_DEVICES               = {os.environ['ROCR_VISIBLE_DEVICES']}")

## 2. Install LeRobot + π₀.₅ Dependencies

LeRobot's `[pi]` extra installs the π₀/π₀.₅ dependencies (JAX stack, OpenPI weights loader, etc.).

Steps:
1. Clone LeRobot from GitHub
2. Install in editable mode with the `pi` extra
3. Install `ffmpeg` for video decoding (TorchCodec requirement)

In [ ]:
import subprocess, sys, os

def run(cmd, **kw):
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if result.returncode != 0:
        print(f"STDERR: {result.stderr[-800:]}")
    else:
        print(result.stdout[-400:] or "(ok)")
    return result.returncode == 0

LEROBOT_DIR = "/workspace/lerobot"

# Clone if not already present
if not os.path.exists(LEROBOT_DIR):
    run(["git", "clone", "https://github.com/huggingface/lerobot.git", LEROBOT_DIR])
else:
    print(f"LeRobot already cloned at {LEROBOT_DIR}")
    run(["git", "-C", LEROBOT_DIR, "pull", "--ff-only"])

# Install LeRobot with pi extras
run([
    sys.executable, "-m", "pip", "install", "-q",
    "--trusted-host", "pypi.org",
    "--trusted-host", "files.pythonhosted.org",
    "-e", f"{LEROBOT_DIR}[pi]",
])

# ffmpeg for TorchCodec video decoding (PyTorch >= 2.10 supports system ffmpeg)
run(["sudo", "apt-get", "install", "-y", "-q", "ffmpeg"])

print("\n✅ Installation complete.")

## 3. Verify LeRobot Installation

In [ ]:
import importlib, sys

def check(module, attr=None):
    try:
        m = importlib.import_module(module)
        ver = getattr(m, "__version__", "?")
        if attr:
            getattr(m, attr)
        print(f"✅ {module} {ver}")
    except Exception as e:
        print(f"❌ {module}: {e}")

check("lerobot")
check("torch")
check("transformers")
check("diffusers")
check("huggingface_hub")

# Verify pi05 policy is registered
try:
    from lerobot.policies.pi05.modeling_pi05 import Pi05Policy
    print("✅ Pi05Policy importable")
except ImportError as e:
    print(f"❌ Pi05Policy not found: {e}")
    print("   Ensure lerobot was installed with: pip install -e '.[pi]'")

import torch
print(f"\nPyTorch CUDA (HIP) available : {torch.cuda.is_available()}")
print(f"BF16 supported               : {torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}")

## 4. Download π₀.₅ Model Weights

Two pretrained checkpoints are available:

| Checkpoint | Description |
|---|---|
| `lerobot/pi05_base` | General base model |
| `lerobot/pi05_libero` | Finetuned on Libero benchmark (97.5% avg) |

Download via `huggingface_hub`. Weights are cached at `~/.cache/huggingface/hub` by default, or at `HF_HOME` if set.

In [ ]:
import os
from huggingface_hub import snapshot_download

MODEL_REPO = "lerobot/pi05_base"   # or "lerobot/pi05_libero"
MODEL_CACHE = "/workspace/pi05_weights"

os.makedirs(MODEL_CACHE, exist_ok=True)

# Check if already downloaded
local_dir = os.path.join(MODEL_CACHE, MODEL_REPO.replace("/", "--"))
if os.path.isdir(local_dir):
    print(f"Weights already cached at: {local_dir}")
else:
    print(f"Downloading {MODEL_REPO} to {MODEL_CACHE} ...")
    snapshot_download(
        repo_id=MODEL_REPO,
        cache_dir=MODEL_CACHE,
    )
    print("Download complete.")

print(f"\nModel repo : {MODEL_REPO}")
print(f"Cache dir  : {MODEL_CACHE}")

## 5. Load π₀.₅ Policy for Inference

Load the pretrained policy from HuggingFace Hub using LeRobot's `from_pretrained` API. We force `device=cuda` and `dtype=bfloat16` for ROCm compatibility.

In [ ]:
import torch
from lerobot.policies.pi05.modeling_pi05 import Pi05Policy

DEVICE = "cuda"  # ROCm exposes AMD GPU as cuda
DTYPE  = torch.bfloat16  # Required — BF16 is the tested precision for π₀.₅

print(f"Loading {MODEL_REPO} on {DEVICE} ({DTYPE}) ...")
print(f"VRAM before load: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

policy = Pi05Policy.from_pretrained(
    MODEL_REPO,
    device=DEVICE,
    dtype=DTYPE,
)
policy.eval()

print(f"\n✅ Policy loaded.")
print(f"VRAM after load : {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
n_params = sum(p.numel() for p in policy.parameters()) / 1e9
print(f"Parameters      : {n_params:.2f} B")

## 6. Run Inference (Dummy Observation)

π₀.₅ expects observations containing:
- **Images** from one or more cameras (RGB, `[B, C, H, W]`, float32 in `[0, 1]`)
- **Robot state** (joint positions, gripper state) as a float tensor
- **Language instruction** as a string

Below we construct a synthetic observation to verify the forward pass on ROCm.

In [ ]:
import torch

DEVICE = "cuda"
DTYPE  = torch.bfloat16

# Synthetic observation — replace with real camera/state data for deployment
batch = {
    # Camera images: [batch, channels, height, width], float32 in [0, 1]
    "observation.images.top": torch.rand(1, 3, 224, 224, device=DEVICE, dtype=DTYPE),
    "observation.images.wrist": torch.rand(1, 3, 224, 224, device=DEVICE, dtype=DTYPE),
    # Robot state: joint positions + gripper (6 DoF arm + 1 gripper = 7 dims)
    "observation.state": torch.zeros(1, 7, device=DEVICE, dtype=DTYPE),
    # Language task instruction
    "task": ["Pick up the red block and place it in the bin."],
}

print("Running forward pass on ROCm...")
with torch.no_grad():
    actions = policy.select_action(batch)

print(f"\n✅ Inference successful.")
print(f"Action tensor shape : {actions.shape}")
print(f"Action dtype        : {actions.dtype}")
print(f"Action device       : {actions.device}")
print(f"Sample actions      : {actions[0, :3].cpu().float().numpy()}")
print(f"VRAM used           : {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

## 7. Finetune π₀.₅ on a Custom Dataset

Use `lerobot-train` CLI to finetune. Two modes:

| Mode | Flag | VRAM | What trains |
|---|---|---|---|
| **Expert only** | `--policy.train_expert_only=true` | ~24 GB | Action expert + projections only |
| **Full finetune** | `--policy.train_expert_only=false` | ~40 GB | All params including VLM |

The cell below constructs and prints the training command. Run it in a terminal or via `subprocess`.

In [ ]:
import shlex

DATASET_REPO   = "your_hf_username/your_dataset"  # ← replace
OUTPUT_DIR     = "/workspace/pi05_finetune"
PRETRAINED     = "lerobot/pi05_base"               # or lerobot/pi05_libero
STEPS          = 3000
BATCH_SIZE     = 32
TRAIN_EXPERT   = "true"   # set false for full finetune (~40 GB VRAM)

cmd = [
    "lerobot-train",
    f"--dataset.repo_id={DATASET_REPO}",
    "--policy.type=pi05",
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=pi05_rocm_finetune",
    f"--policy.pretrained_path={PRETRAINED}",
    # ROCm: compile_model uses torch.compile — works on ROCm 6+
    "--policy.compile_model=true",
    # Gradient checkpointing cuts activation memory ~4x on ROCm
    "--policy.gradient_checkpointing=true",
    "--policy.dtype=bfloat16",
    "--policy.device=cuda",              # ROCm exposes as 'cuda'
    "--policy.freeze_vision_encoder=false",
    f"--policy.train_expert_only={TRAIN_EXPERT}",
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--wandb.enable=false",
]

print("Training command:")
print("\n".join(f"    {part} \\" for part in cmd))
print()
print("To run: copy the command above into a terminal, or set run_training=True below.")

In [ ]:
import subprocess, os

run_training = False  # ← set True to actually launch training

if run_training:
    env = os.environ.copy()
    # ROCm tuning for training
    env["TORCH_USE_HIP_DSA"] = "1"
    env["FLASH_ATTENTION_TRITON_AMD_ENABLE"] = "TRUE"
    env["PYTORCH_TUNABLEOP_ENABLED"] = "1"
    env["ROCR_VISIBLE_DEVICES"] = "0"

    print("Launching training...")
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    # Stream first 200 lines then detach
    for i, line in enumerate(proc.stdout):
        print(line, end="")
        if i >= 200:
            print("... (output truncated, training running in background)")
            break
else:
    print("Skipped — set run_training=True to launch.")

## 8. (Optional) Relative Actions

π₀.₅ can predict *offsets* relative to the current robot state instead of absolute joint targets. This improves stability for some setups.

**Step 1** — recompute dataset stats in relative space:

In [ ]:
# Run this in a terminal to recompute stats before training with relative actions
relative_actions_cmd = """\
lerobot-edit-dataset \\
    --repo_id your_hf_username/your_dataset \\
    --operation.type recompute_stats \\
    --operation.relative_action true \\
    --operation.chunk_size 50 \\
    --operation.relative_exclude_joints "['gripper']" \\
    --push_to_hub true
"""
print(relative_actions_cmd)

# Equivalent Python API:
print("Or in Python:")
print("""
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.datasets.dataset_tools import recompute_stats

dataset = LeRobotDataset("your_hf_username/your_dataset")
recompute_stats(dataset, relative_action=True, chunk_size=50,
                relative_exclude_joints=["gripper"])
dataset.push_to_hub()
""")

In [ ]:
# Training command with relative actions enabled
relative_train_cmd = """\
lerobot-train \\
    --dataset.repo_id=your_hf_username/your_dataset \\
    --policy.type=pi05 \\
    --policy.pretrained_path=lerobot/pi05_base \\
    --policy.use_relative_actions=true \\
    --policy.relative_exclude_joints='["gripper"]' \\
    --policy.dtype=bfloat16 \\
    --policy.device=cuda \\
    --policy.compile_model=true \\
    --policy.gradient_checkpointing=true \\
    --steps=3000 \\
    --batch_size=32
"""
print(relative_train_cmd)

## 9. Libero Benchmark (Reference)

To reproduce the published results, finetune `lerobot/pi05_libero` for 6k steps on the Libero dataset:

| Benchmark | LeRobot | OpenPI reference |
|---|---|---|
| Libero Spatial | 97.0% | 98.8% |
| Libero Object | 99.0% | 98.2% |
| Libero Goal | 98.0% | 98.0% |
| Libero 10 | 96.0% | 92.4% |
| **Average** | **97.5%** | **96.85%** |

See the [LeRobot Libero guide](https://huggingface.co/docs/lerobot/libero) for full reproduction steps.

In [ ]:
libero_cmd = """\
lerobot-train \\
    --dataset.repo_id=lerobot/libero_10 \\
    --policy.type=pi05 \\
    --policy.pretrained_path=lerobot/pi05_libero \\
    --policy.compile_model=true \\
    --policy.gradient_checkpointing=true \\
    --policy.dtype=bfloat16 \\
    --policy.device=cuda \\
    --policy.freeze_vision_encoder=false \\
    --policy.train_expert_only=false \\
    --steps=6000 \\
    --batch_size=32 \\
    --output_dir=/workspace/pi05_libero_finetune
"""
print(libero_cmd)

---

## Tips & Troubleshooting on AMD ROCm

| Issue | Fix |
|---|---|
| `RuntimeError: HIP error: no kernel image is available` | Set `HSA_OVERRIDE_GFX_VERSION=11.0.0` for RDNA3 (gfx1100) |
| OOM during full finetune | Use `--policy.train_expert_only=true` (~24 GB) or reduce batch size |
| `torch.compile` crash on ROCm | Add `--policy.compile_model=false` to disable; or upgrade ROCm to 6.x |
| BF16 errors | Verify `torch.cuda.is_bf16_supported()` returns True; RDNA3+ and CDNA2+ support native BF16 |
| Flash Attention not active | Confirm `FLASH_ATTENTION_TRITON_AMD_ENABLE=TRUE` is set before importing torch |
| Slow first step | Normal — PyTorch ROCm compiles Triton kernels on first run; subsequent steps are faster |
| `lerobot-train` not found | Ensure LeRobot was installed with `pip install -e '.[pi]'` and the venv is active |

## References

- [π₀.₅ LeRobot Documentation](https://huggingface.co/docs/lerobot/en/pi05)
- [OpenPI Repository (Physical Intelligence)](https://github.com/Physical-Intelligence/openpi)
- [π₀.₅ Blog Post](https://www.physicalintelligence.company/blog/pi05)
- [lerobot/pi05_base on HuggingFace](https://huggingface.co/lerobot/pi05_base)
- [lerobot/pi05_libero on HuggingFace](https://huggingface.co/lerobot/pi05_libero)
- [LeRobot GitHub](https://github.com/huggingface/lerobot)
- [ROCm Documentation](https://rocm.docs.amd.com)
- [learn-world-model Project](https://github.com/qiwang067/learn-world-model)